In [3]:
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torchvision import transforms
from PIL import Image

def dice_coefficient(pred, target, epsilon=1e-6):
    pred = torch.sigmoid(pred)      
    pred = (pred > 0.5).float()       
    intersection = (pred * target).sum(dim=(1, 2, 3))
    union = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3)) + epsilon
    dice = (2.0 * intersection + epsilon) / union
    return dice.mean()


seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ======================
# 3) تعریف دیتاست
# ======================
class CorneaDataset(Dataset):
    def __init__(self, images_list, masks_list, transform=None):
        self.images_list = images_list
        self.masks_list = masks_list
        self.transform = transform

    def __len__(self):
        return len(self.images_list)

    def __getitem__(self, idx):
        img_path = self.images_list[idx]
        mask_path = self.masks_list[idx]

        image = Image.open(img_path).convert("RGB")  # اگر تصاویر رنگی‌اند
        mask = Image.open(mask_path).convert("L")    # ماسک معمولاً خاکستری یا باینری

        if self.transform:
            image = self.transform(image)
            mask = self.transform(mask)

        # باینری کردن ماسک (اگر داده‌های شما 0 و 255 هستند)
        mask = (mask > 0.5).float()

        return image, mask

# ======================
# 4) آدرس فولدر تصاویر
# ======================
images_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\my_work\segment\images"
labels_dir = r"C:\Users\Meta Pc\PycharmProjects\pythonProject\myproject\payanName\segment\corneaLabels"

images_list = sorted(glob.glob(os.path.join(images_dir, "*.*")))
masks_list = sorted(glob.glob(os.path.join(labels_dir, "*.*")))
assert len(images_list) == len(masks_list), "تعداد تصاویر با ماسک‌ها برابر نیست."

# ======================
# 5) تقسیم داده‌ها: 80% Train, 20% Test
# ======================
total_size = len(images_list)
train_size = int(total_size * 0.8)
test_size  = total_size - train_size

train_images = images_list[:train_size]
train_masks  = masks_list[:train_size]
test_images  = images_list[train_size:]
test_masks   = masks_list[train_size:]

# ======================
# 6) تعریف ترنسفورم‌ها
# ======================
# ترنسفورم برای داده‌های تست
transform_test = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ترنسفورم پایه برای داده‌های آموزش
transform_train_base = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor()
])

# ترنسفورم آگومنت‌شده برای داده‌های آموزش
transform_train_aug = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ToTensor()
])

# ======================
# 7) ساخت دیتاست
# ======================
# دیتاست اصلی
train_dataset_base = CorneaDataset(train_images, train_masks, transform=transform_train_base)
# دیتاست آگومنت‌شده
train_dataset_aug  = CorneaDataset(train_images, train_masks, transform=transform_train_aug)
# ترکیب دیتاست اصلی و آگومنت شده
train_dataset      = ConcatDataset([train_dataset_base, train_dataset_aug])

test_dataset       = CorneaDataset(test_images, test_masks, transform=transform_test)

# ======================
# 8) تعریف دیتالودر
# ======================
batch_size = 2
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, num_workers=0)

# ======================
# 9) تعریف بلوک‌های کمکی: conv_block و up_conv
# ======================
def conv_block(in_channels, out_channels):
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(inplace=True)
    )

def up_conv(in_channels, out_channels):
    return nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)

# ======================
# 10) تعریف مدل TransUNet
# ======================
class TransUNet(nn.Module):
    def __init__(self, in_channels=3, out_channels=1, img_size=256, patch_size=16, emb_dim=512, num_transformer_layers=4, nhead=8):
        super(TransUNet, self).__init__()
        # Encoder: استخراج ویژگی‌ها با استفاده از بلوک‌های CNN
        self.encoder1 = conv_block(in_channels, 64)     # خروجی: [B, 64, 256, 256]
        self.pool1    = nn.MaxPool2d(2)                   # [B, 64, 128, 128]
        self.encoder2 = conv_block(64, 128)               # [B, 128, 128, 128]
        self.pool2    = nn.MaxPool2d(2)                   # [B, 128, 64, 64]
        self.encoder3 = conv_block(128, 256)              # [B, 256, 64, 64]
        self.pool3    = nn.MaxPool2d(2)                   # [B, 256, 32, 32]
        self.encoder4 = conv_block(256, 512)              # [B, 512, 32, 32]
        self.pool4    = nn.MaxPool2d(2)                   # [B, 512, 16, 16]
        
        # Transformer branch روی خروجی عمیق encoder
        # تعداد tokenها = (16 x 16) = 256
        self.num_tokens = (img_size // 16) * (img_size // 16)
        self.emb_dim = emb_dim  # انتظار می‌رود برابر 512 (همان تعداد کانال encoder4)
        self.pos_embed = nn.Parameter(torch.randn(1, self.num_tokens, emb_dim))
        encoder_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_transformer_layers)
        
        # Decoder: بازسازی نقشه segmentation
        self.up1 = up_conv(emb_dim, 512)                  # آپ‌سمپل: [B, 512, 16, 16] -> [B, 512, 32, 32]
        self.decoder1 = conv_block(512+512, 512)          # ادغام با e4
        
        self.up2 = up_conv(512, 256)                      # [B, 256, 32, 32] -> [B, 256, 64, 64]
        self.decoder2 = conv_block(256+256, 256)          # ادغام با e3
        
        self.up3 = up_conv(256, 128)                      # [B, 128, 64, 64] -> [B, 128, 128, 128]
        self.decoder3 = conv_block(128+128, 128)          # ادغام با e2
        
        self.up4 = up_conv(128, 64)                       # [B, 64, 128, 128] -> [B, 64, 256, 256]
        self.decoder4 = conv_block(64+64, 64)             # ادغام با e1
        
        self.final_conv = nn.Conv2d(64, out_channels, kernel_size=1)
        
    def forward(self, x):
        # Encoder
        e1 = self.encoder1(x)       # [B, 64, 256, 256]
        p1 = self.pool1(e1)         # [B, 64, 128, 128]
        e2 = self.encoder2(p1)      # [B, 128, 128, 128]
        p2 = self.pool2(e2)         # [B, 128, 64, 64]
        e3 = self.encoder3(p2)      # [B, 256, 64, 64]
        p3 = self.pool3(e3)         # [B, 256, 32, 32]
        e4 = self.encoder4(p3)      # [B, 512, 32, 32]
        p4 = self.pool4(e4)         # [B, 512, 16, 16]
        
        # Transformer branch
        B, C, H, W = p4.shape      # انتظار: C=512, H=W=16
        x_tr = p4.view(B, C, H*W).permute(0, 2, 1)   # [B, 256, 512]
        x_tr = x_tr + self.pos_embed                # افزودن positional encoding
        # nn.TransformerEncoder انتظار ورودی با ابعاد [S, B, D] دارد
        x_tr = self.transformer(x_tr.permute(1, 0, 2))  # [256, B, 512]
        x_tr = x_tr.permute(1, 0, 2)                   # [B, 256, 512]
        x_tr = x_tr.permute(0, 2, 1).view(B, C, H, W)   # [B, 512, 16, 16]
        
        # Decoder
        d1 = self.up1(x_tr)                         # [B, 512, 32, 32]
        d1 = torch.cat([d1, e4], dim=1)              # [B, 1024, 32, 32]
        d1 = self.decoder1(d1)                      # [B, 512, 32, 32]
        
        d2 = self.up2(d1)                           # [B, 256, 64, 64]
        d2 = torch.cat([d2, e3], dim=1)              # [B, 512, 64, 64]
        d2 = self.decoder2(d2)                      # [B, 256, 64, 64]
        
        d3 = self.up3(d2)                           # [B, 128, 128, 128]
        d3 = torch.cat([d3, e2], dim=1)              # [B, 256, 128, 128]
        d3 = self.decoder3(d3)                      # [B, 128, 128, 128]
        
        d4 = self.up4(d3)                           # [B, 64, 256, 256]
        d4 = torch.cat([d4, e1], dim=1)              # [B, 128, 256, 256]
        d4 = self.decoder4(d4)                      # [B, 64, 256, 256]
        
        out = self.final_conv(d4)                   # [B, out_channels, 256, 256]
        return out
    
    

In [4]:
# ======================
# 11) تنظیم دستگاه و مدل، معیار و بهینه‌ساز
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransUNet(in_channels=3, out_channels=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# ======================
# 12) توابع آموزش و اعتبارسنجی
# ======================
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    epoch_loss = 0.0
    epoch_dice = 0.0
    for images, masks in dataloader:
        images = images.to(device)
        masks = masks.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()
        dice_score = dice_coefficient(outputs, masks)
        epoch_loss += loss.item()
        epoch_dice += dice_score.item()
    return epoch_loss / len(dataloader), epoch_dice / len(dataloader)

def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    val_loss = 0.0
    val_dice = 0.0
    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            loss = criterion(outputs, masks)
            dice_score = dice_coefficient(outputs, masks)
            val_loss += loss.item()
            val_dice += dice_score.item()
    return val_loss / len(dataloader), val_dice / len(dataloader)

# ======================
# 13) حلقه آموزش
# ======================
num_epochs = 20
for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, test_loader, criterion, device)  # استفاده از test_loader به عنوان validation در اینجا
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 14) ذخیره مدل
# ======================
torch.save(model.state_dict(), "transunet_cornea.pth")


In [5]:
batch_size = 4

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransUNet(in_channels=3, out_channels=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 20
for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, test_loader, criterion, device)  # استفاده از test_loader به عنوان validation در اینجا
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 14) ذخیره مدل
# ======================
# torch.save(model.state_dict(), "transunet_cornea.pth")


In [6]:
batch_size = 8
# ======================
# 11) تنظیم دستگاه و مدل، معیار و بهینه‌ساز
# ======================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TransUNet(in_channels=3, out_channels=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)



# ======================
# 13) حلقه آموزش
# ======================
num_epochs = 20
for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, test_loader, criterion, device)  # استفاده از test_loader به عنوان validation در اینجا
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

# ======================
# 14) ذخیره مدل
# ======================
# torch.save(model.state_dict(), "transunet_cornea.pth")


In [7]:
batch_size = 16
num_epochs = 20
for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, test_loader, criterion, device)  # استفاده از test_loader به عنوان validation در اینجا
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")

In [8]:
batch_size = 32
num_epochs = 20
for epoch in range(num_epochs):
    train_loss, train_dice = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_dice = validate_one_epoch(model, test_loader, criterion, device)  # استفاده از test_loader به عنوان validation در اینجا
    print(f"Epoch [{epoch+1}/{num_epochs}] - Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f} "
          f"- Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}")